# EDA-04 · Web Traffic vs Revenue
Sessions→orders funnel, bounce_rate trend, traffic_source contribution, seasonality.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
SRC_COLORS   = {'organic_search':'#4C72B0','paid_search':'#DD8452','social_media':'#55A868',
                 'email_campaign':'#C44E52','referral':'#9467BD','direct':'#8C564B'}

web    = pd.read_csv('web_traffic.csv',  parse_dates=['date'])
orders = pd.read_csv('orders.csv',       parse_dates=['order_date'])
items  = pd.read_csv('order_items.csv',  low_memory=False)
sales  = pd.read_csv('sales.csv',        parse_dates=['Date'])

# Daily orders & revenue
daily_orders = orders.groupby('order_date').agg(
    n_orders = ('order_id','count')).reset_index().rename(columns={'order_date':'date'})

daily_rev = (
    items.merge(orders[['order_id','order_date']], on='order_id', how='left')
    .groupby('order_date')
    .agg(revenue=('unit_price', lambda x: (x * items.loc[x.index,'quantity']).sum()))
    .reset_index().rename(columns={'order_date':'date'})
)

# Daily orders by source
daily_src_orders = (
    orders.groupby(['order_date','order_source'])['order_id']
    .count().reset_index()
    .rename(columns={'order_date':'date','order_id':'n_orders','order_source':'traffic_source'})
)

# Merge web + orders (overlap 2013-2022)
df = web.merge(daily_orders, on='date', how='left')
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month
df['conv_rate'] = df['n_orders'] / df['sessions']

# Join with order source breakdown
df_src = web.merge(daily_src_orders, on=['date','traffic_source'], how='left')
df_src['conv_rate'] = df_src['n_orders'] / df_src['sessions']

print(f'web_traffic rows  : {len(web):,}  ({web["date"].min().date()} -> {web["date"].max().date()})')
print(f'Overlap with orders: {df["n_orders"].notna().sum():,} days')
print(f'Traffic sources    : {web["traffic_source"].unique().tolist()}')
df.head(3)


## 1. Overview: Web Traffic & Orders

In [ ]:
print('=== Web Traffic Summary ===')
print(web[['sessions','unique_visitors','page_views','bounce_rate','avg_session_duration_sec']].describe().round(2).to_string())

print()
print(f'Total sessions (2013-2022)       : {web["sessions"].sum():,.0f}')
print(f'Total unique_visitors            : {web["unique_visitors"].sum():,.0f}')
print(f'Avg daily sessions               : {web["sessions"].mean():,.0f}')
print(f'Avg daily orders (overlap period): {df["n_orders"].mean():.1f}')
print(f'Overall conversion rate          : {df["n_orders"].sum()/df["sessions"].sum()*100:.4f}%')

## 2. Sessions → Orders Funnel

In [ ]:
# Funnel by traffic_source
funnel = df_src.groupby('traffic_source').agg(
    total_sessions  = ('sessions','sum'),
    total_visitors  = ('unique_visitors','sum'),
    total_orders    = ('n_orders','sum'),
).fillna(0).reset_index()
funnel['conv_rate_%']      = funnel['total_orders'] / funnel['total_sessions'] * 100
funnel['visitor_rate_%']   = funnel['total_visitors'] / funnel['total_sessions'] * 100
funnel = funnel.sort_values('conv_rate_%', ascending=False)
print('Funnel by traffic source:')
print(funnel.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Conversion rate bar
colors = [SRC_COLORS.get(s,'grey') for s in funnel['traffic_source']]
bars = axes[0].barh(funnel['traffic_source'], funnel['conv_rate_%'], color=colors)
axes[0].set_xlabel('Conversion rate %  (orders / sessions)')
axes[0].set_title('Conversion rate by traffic source')
for bar, v in zip(bars, funnel['conv_rate_%']):
    axes[0].text(bar.get_width()+0.002, bar.get_y()+bar.get_height()/2,
                 f'{v:.4f}%', va='center', fontsize=8)

# Funnel waterfall per source (sessions → visitors → orders scaled)
x = range(len(funnel))
w = 0.25
axes[1].bar([i-w for i in x], funnel['total_sessions']/1e6,  width=w, label='Sessions (M)',  color='#4C72B0', alpha=0.85)
axes[1].bar([i   for i in x], funnel['total_visitors']/1e6,  width=w, label='Visitors (M)',  color='#55A868', alpha=0.85)
axes[1].bar([i+w for i in x], funnel['total_orders']/1e3,    width=w, label='Orders (K)',    color='#C44E52', alpha=0.85)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(funnel['traffic_source'], rotation=20, ha='right', fontsize=8)
axes[1].set_ylabel('Volume (M/K)')
axes[1].set_title('Sessions vs Visitors vs Orders by source')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3. Conversion Rate Trend by Year

In [ ]:
# Yearly overall
yr_conv = df.groupby('year').agg(
    sessions = ('sessions','sum'),
    orders   = ('n_orders','sum'),
).reset_index()
yr_conv['conv_rate_%'] = yr_conv['orders'] / yr_conv['sessions'] * 100

# Yearly by source
yr_src = df_src.groupby(['year','traffic_source']).agg(
    sessions = ('sessions','sum'),
    orders   = ('n_orders','sum'),
).reset_index().fillna(0)
yr_src['conv_rate_%'] = yr_src['orders'] / yr_src['sessions'] * 100

print('Overall conversion rate by year:')
print(yr_conv[['year','sessions','orders','conv_rate_%']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(yr_conv['year'], yr_conv['conv_rate_%'], marker='o', color='#4C72B0', lw=2)
axes[0].fill_between(yr_conv['year'], yr_conv['conv_rate_%'], alpha=0.15, color='#4C72B0')
axes[0].set_xlabel('Year'); axes[0].set_ylabel('Conversion rate %')
axes[0].set_title('Overall conversion rate by year')
for _, row in yr_conv.iterrows():
    axes[0].text(row['year'], row['conv_rate_%']+0.001, f'{row["conv_rate_%"]:.4f}%', ha='center', fontsize=7)

for src, grp in yr_src.groupby('traffic_source'):
    axes[1].plot(grp['year'], grp['conv_rate_%'], marker='o', ms=4,
                 label=src, color=SRC_COLORS.get(src,'grey'), lw=1.5)
axes[1].set_xlabel('Year'); axes[1].set_ylabel('Conversion rate %')
axes[1].set_title('Conversion rate by source & year')
axes[1].legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

## 4. Bounce Rate Trend

In [ ]:
# Monthly bounce rate
df['month_period'] = df['date'].dt.to_period('M')
monthly_bounce = df.groupby('month_period').agg(
    avg_bounce = ('bounce_rate','mean'),
    avg_sessions = ('sessions','mean'),
    avg_orders = ('n_orders','mean'),
).reset_index()
monthly_bounce['month_ts'] = monthly_bounce['month_period'].dt.to_timestamp()

# By source
src_bounce = web.groupby('traffic_source')['bounce_rate'].agg(['mean','std','min','max']).round(6)
print('Bounce rate by traffic source:')
print(src_bounce.to_string())

r_bounce_orders = df['bounce_rate'].corr(df['n_orders'])
r_bounce_conv   = df['bounce_rate'].corr(df['conv_rate'])
print(f'\nCorr(bounce_rate, n_orders)   : {r_bounce_orders:.4f}')
print(f'Corr(bounce_rate, conv_rate)  : {r_bounce_conv:.4f}')

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

axes[0].plot(monthly_bounce['month_ts'], monthly_bounce['avg_bounce']*100,
             color='#C44E52', lw=1.2)
axes[0].set_ylabel('Avg bounce rate %'); axes[0].set_title('Monthly bounce rate trend')

axes[1].plot(monthly_bounce['month_ts'], monthly_bounce['avg_sessions'],
             color='#4C72B0', lw=1.2, label='Sessions')
ax2 = axes[1].twinx()
ax2.plot(monthly_bounce['month_ts'], monthly_bounce['avg_orders'],
         color='#55A868', lw=1.2, linestyle='--', label='Orders')
axes[1].set_ylabel('Avg sessions'); ax2.set_ylabel('Avg orders')
axes[1].set_title('Monthly sessions vs orders')
lines1, lbl1 = axes[1].get_legend_handles_labels()
lines2, lbl2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1+lines2, lbl1+lbl2, loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

fig2, ax3 = plt.subplots(figsize=(9, 4))
for src, grp in web.groupby('traffic_source'):
    mo = grp.groupby(grp['date'].dt.to_period('M'))['bounce_rate'].mean().reset_index()
    mo['ts'] = mo['date'].dt.to_timestamp()
    ax3.plot(mo['ts'], mo['bounce_rate']*100, lw=1, alpha=0.8,
             label=src, color=SRC_COLORS.get(src,'grey'))
ax3.set_ylabel('Bounce rate %'); ax3.set_title('Bounce rate by traffic source over time')
ax3.legend(fontsize=8); plt.tight_layout(); plt.show()

## 5. Traffic Source Contribution (Sessions vs Revenue)

In [ ]:
# Sessions share by source
src_sess = web.groupby('traffic_source')['sessions'].sum()
src_sess_pct = src_sess / src_sess.sum() * 100

# Orders share by source (from orders.csv)
src_ord  = orders.groupby('order_source')['order_id'].count().rename(index=lambda x: x)
src_ord_pct = src_ord / src_ord.sum() * 100

# Revenue share by source
rev_by_src = (
    items.merge(orders[['order_id','order_source']], on='order_id', how='left')
    .assign(rev_line=lambda d: d['quantity']*d['unit_price'])
    .groupby('order_source')['rev_line'].sum()
)
rev_by_src_pct = rev_by_src / rev_by_src.sum() * 100

contrib = pd.DataFrame({
    'sessions_%' : src_sess_pct,
    'orders_%'   : src_ord_pct,
    'revenue_%'  : rev_by_src_pct,
}).fillna(0).sort_values('revenue_%', ascending=False)
print('Traffic source contribution:')
print(contrib.round(2).to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, title in zip(axes,
    ['sessions_%','orders_%','revenue_%'],
    ['Sessions share %','Orders share %','Revenue share %']):
    colors = [SRC_COLORS.get(s,'grey') for s in contrib.index]
    wedges, texts, autotexts = ax.pie(
        contrib[col], labels=contrib.index, autopct='%1.1f%%',
        colors=colors, startangle=140,
        textprops={'fontsize':7})
    ax.set_title(title)
plt.tight_layout(); plt.show()

## 6. Source Share Trend Over Time

In [ ]:
# Yearly sessions share
yr_src_sess = (
    web.groupby(['year','traffic_source'])['sessions'].sum()
    .unstack(fill_value=0)
)
yr_src_sess_pct = yr_src_sess.div(yr_src_sess.sum(axis=1), axis=0) * 100

# Yearly orders share
orders['year'] = orders['order_date'].dt.year
yr_src_ord = (
    orders.groupby(['year','order_source'])['order_id'].count()
    .unstack(fill_value=0)
)
yr_src_ord_pct = yr_src_ord.div(yr_src_ord.sum(axis=1), axis=0) * 100

print('Yearly sessions share %:')
print(yr_src_sess_pct.round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

src_list = yr_src_sess_pct.columns.tolist()
colors_list = [SRC_COLORS.get(s,'grey') for s in src_list]

yr_src_sess_pct.plot.area(ax=axes[0], color=colors_list, alpha=0.8, linewidth=0)
axes[0].set_xlabel('Year'); axes[0].set_ylabel('Share %')
axes[0].set_title('Web sessions share by source (stacked area)')
axes[0].legend(fontsize=7, loc='upper left')

src_list2 = yr_src_ord_pct.columns.tolist()
colors_list2 = [SRC_COLORS.get(s,'grey') for s in src_list2]
yr_src_ord_pct.plot.area(ax=axes[1], color=colors_list2, alpha=0.8, linewidth=0)
axes[1].set_xlabel('Year'); axes[1].set_ylabel('Share %')
axes[1].set_title('Orders share by source (stacked area)')
axes[1].legend(fontsize=7, loc='upper left')
plt.tight_layout(); plt.show()

## 7. Seasonality — Web Traffic vs Sales Revenue

In [ ]:
# Monthly seasonal profile (normalized 0-1 per metric)
df['month'] = df['date'].dt.month
sales['month'] = sales['Date'].dt.month

monthly_web = df.groupby('month').agg(
    avg_sessions  = ('sessions','mean'),
    avg_visitors  = ('unique_visitors','mean'),
    avg_orders    = ('n_orders','mean'),
    avg_bounce    = ('bounce_rate','mean'),
    avg_dur       = ('avg_session_duration_sec','mean'),
).reset_index()

monthly_sales = sales.groupby('month').agg(
    avg_revenue = ('Revenue','mean'),
    avg_cogs    = ('COGS','mean'),
).reset_index()

sea = monthly_web.merge(monthly_sales, on='month')

def norm(s): return (s - s.min()) / (s.max() - s.min())

print('Monthly seasonal profile (normalized):')
sea_show = sea[['month','avg_sessions','avg_orders','avg_revenue']].copy()
for c in ['avg_sessions','avg_orders','avg_revenue']:
    sea_show[c+'_norm'] = norm(sea[c])
print(sea_show.round(3).to_string(index=False))

print('\nPearson correlations (12 monthly avg points):')
for col in ['avg_sessions','avg_visitors','avg_bounce','avg_dur','avg_orders']:
    r = sea[col].corr(sea['avg_revenue'])
    print(f'  {col:<30}: r = {r:.4f}')

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

for metric, color, label in [
    ('avg_sessions','#4C72B0','Sessions'),
    ('avg_orders',  '#55A868','Orders'),
]:
    axes[0].plot(sea['month'], norm(sea[metric]), marker='o', label=label, color=color)
axes[0].plot(sea['month'], norm(sea['avg_revenue']), marker='s', lw=2,
             label='Revenue', color='#C44E52', linestyle='--')
axes[0].set_ylabel('Normalized value (0–1)')
axes[0].set_title('Seasonal pattern: Sessions / Orders / Revenue (normalized)')
axes[0].set_xticks(range(1,13)); axes[0].set_xticklabels(MONTH_LABELS)
axes[0].legend(fontsize=9)

axes[1].plot(sea['month'], sea['avg_bounce']*100, marker='o', color='#9467BD', label='Bounce rate %')
ax2b = axes[1].twinx()
ax2b.plot(sea['month'], sea['avg_dur'], marker='s', color='#8C564B',
          linestyle='--', label='Avg session duration (s)')
axes[1].set_ylabel('Bounce rate %'); ax2b.set_ylabel('Duration (s)')
axes[1].set_title('Seasonal bounce rate & session duration')
axes[1].set_xticks(range(1,13)); axes[1].set_xticklabels(MONTH_LABELS)
l1,lb1 = axes[1].get_legend_handles_labels()
l2,lb2 = ax2b.get_legend_handles_labels()
axes[1].legend(l1+l2, lb1+lb2, fontsize=8)
plt.tight_layout(); plt.show()

## 8. Year-over-Year: Sessions Growth vs Revenue Growth

In [ ]:
yr_web = web.groupby('year').agg(
    total_sessions  = ('sessions','sum'),
    total_visitors  = ('unique_visitors','sum'),
    avg_bounce      = ('bounce_rate','mean'),
    avg_duration    = ('avg_session_duration_sec','mean'),
).reset_index()

yr_sales = sales.groupby(sales['Date'].dt.year)[['Revenue','COGS']].sum().reset_index()
yr_sales.columns = ['year','Revenue','COGS']

yr = yr_web.merge(yr_sales, on='year', how='inner')
yr['yoy_sessions_%'] = yr['total_sessions'].pct_change() * 100
yr['yoy_revenue_%']  = yr['Revenue'].pct_change() * 100

print('YoY growth — sessions vs revenue:')
print(yr[['year','total_sessions','Revenue','yoy_sessions_%','yoy_revenue_%']].round(1).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax_r = axes[0].twinx() if True else None
axes[0].bar(yr['year']-0.2, yr['total_sessions']/1e6, width=0.4,
            label='Sessions (M)', color='#4C72B0', alpha=0.8)
ax_r = axes[0].twinx()
ax_r.bar(yr['year']+0.2, yr['Revenue']/1e9, width=0.4,
         label='Revenue (B VND)', color='#C44E52', alpha=0.8)
axes[0].set_ylabel('Sessions (M)'); ax_r.set_ylabel('Revenue (B VND)')
axes[0].set_title('Annual sessions vs revenue')
axes[0].set_xlabel('Year')
h1,l1 = axes[0].get_legend_handles_labels()
h2,l2 = ax_r.get_legend_handles_labels()
axes[0].legend(h1+h2, l1+l2, fontsize=8, loc='upper left')

yr2 = yr.dropna(subset=['yoy_sessions_%','yoy_revenue_%'])
axes[1].plot(yr2['year'], yr2['yoy_sessions_%'], marker='o', label='Sessions YoY %', color='#4C72B0')
axes[1].plot(yr2['year'], yr2['yoy_revenue_%'],  marker='s', label='Revenue YoY %',  color='#C44E52', linestyle='--')
axes[1].axhline(0, color='black', lw=0.8, linestyle=':')
axes[1].set_xlabel('Year'); axes[1].set_ylabel('YoY growth %')
axes[1].set_title('YoY growth: sessions vs revenue')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 9. Daily Scatter: Sessions vs Revenue

In [ ]:
df_rev = df.merge(sales.rename(columns={'Date':'date'}), on='date', how='inner')
df_rev['year'] = df_rev['date'].dt.year

r_daily = df_rev['sessions'].corr(df_rev['Revenue'])
print(f'Daily Pearson correlation sessions vs Revenue: r = {r_daily:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sc = axes[0].scatter(df_rev['sessions']/1e3, df_rev['Revenue']/1e6,
                     c=df_rev['year'], cmap='viridis', alpha=0.3, s=8)
plt.colorbar(sc, ax=axes[0], label='Year')
axes[0].set_xlabel('Daily sessions (K)'); axes[0].set_ylabel('Daily Revenue (M VND)')
axes[0].set_title(f'Daily sessions vs Revenue  (r={r_daily:.3f})')

# Monthly scatter
monthly_both = df_rev.groupby(df_rev['date'].dt.to_period('M')).agg(
    sessions=('sessions','sum'), Revenue=('Revenue','sum')).reset_index()
r_mo = monthly_both['sessions'].corr(monthly_both['Revenue'])
axes[1].scatter(monthly_both['sessions']/1e6, monthly_both['Revenue']/1e6,
                alpha=0.5, s=25, color='#4C72B0')
axes[1].set_xlabel('Monthly sessions (M)'); axes[1].set_ylabel('Monthly Revenue (M VND)')
axes[1].set_title(f'Monthly sessions vs Revenue  (r={r_mo:.3f})')
plt.tight_layout(); plt.show()

## Summary

In [ ]:
print('=== Web Traffic vs Revenue Summary ===')
print(f'Period          : {web["date"].min().date()} -> {web["date"].max().date()}')
print(f'Avg sessions/day: {web["sessions"].mean():,.0f}')
print(f'Overall conv.   : {df["n_orders"].sum()/df["sessions"].sum()*100:.4f}%  (orders/sessions)')
print()
print('Conversion rate by source:')
for _, row in funnel.iterrows():
    print(f'  {row["traffic_source"]:<18}: {row["conv_rate_%"]:.4f}%')
print()
print(f'Seasonal corr sessions-revenue (monthly): r = {sea["avg_sessions"].corr(sea["avg_revenue"]):.4f}')
print(f'Daily   corr sessions-revenue           : r = {r_daily:.4f}')
print()
print('Sessions share vs Revenue share (top 3 sources):')
for src in contrib.head(3).index:
    print(f'  {src:<18}: sessions={contrib.loc[src,"sessions_%"]:.1f}%  revenue={contrib.loc[src,"revenue_%"]:.1f}%')